# Day 18 — Production Deployment, Dockerization & Cloud-Ready Architecture

Package the platform (the Day-14 FastAPI app + all 13 layers) into a reproducible container + Compose
stack, with environment config, centralized logging, health probes, deploy scripts, a cloud blueprint,
and a security checklist.

**Two honest constraints shape this notebook:**
1. **Docker is not installed here**, so we *statically validate* the Dockerfile / compose / scripts and
   boot the app via TestClient — `docker build`/`run` is the operator's step (never claimed as run here).
2. **One container, FAISS in-process** — there is no separate vector-DB service; the whole platform is
   one process, and the 900 MB of trained artifacts are **mounted**, not baked into the image.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # before any faiss/torch import (TestClient loads both)
import sys, json, subprocess, yaml
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
print("repo:", os.getcwd())

repo: /Users/Yash/AI-Ventutre-Intelligence-Platform


## 1 — The deployment artifacts are all present

Every file Phases B–J produced.

In [2]:
files = ["Dockerfile", ".dockerignore", "docker-compose.yml", ".env.example",
         "scripts/build.sh", "scripts/run.sh", "scripts/stop.sh", "scripts/deploy.sh",
         "deployment/nginx.conf", "deployment/cloud_architecture.md",
         "deployment/security_checklist.md", "deployment/deployment_guide.md"]
for f in files:
    print(("  ok  " if os.path.exists(f) else "  MISSING ") + f)
assert all(os.path.exists(f) for f in files)

  ok  Dockerfile
  ok  .dockerignore
  ok  docker-compose.yml
  ok  .env.example
  ok  scripts/build.sh
  ok  scripts/run.sh
  ok  scripts/stop.sh
  ok  scripts/deploy.sh
  ok  deployment/nginx.conf
  ok  deployment/cloud_architecture.md
  ok  deployment/security_checklist.md
  ok  deployment/deployment_guide.md


## 2 — docker-compose: ONE app service, FAISS in-process (no vector-DB service)

The stack is one `api` container; artifacts/data are mounted **read-only**; a healthcheck probes
`/health`. nginx and mlflow-ui are optional (profiles).

In [3]:
d = yaml.safe_load(open("docker-compose.yml"))
svcs = d["services"]
always_on = [s for s, v in svcs.items() if not v.get("profiles")]
forbidden = [s for s in svcs if any(x in s.lower() for x in ("vector", "chroma", "redis", "db"))]
api = svcs["api"]
print("services         :", list(svcs))
print("always-on        :", always_on, "(should be just 'api')")
print("forbidden svcs   :", forbidden or "none")
print("api volumes      :", api["volumes"])
print("api healthcheck  :", api["healthcheck"]["test"][-1][:60], "...")
assert always_on == ["api"] and not forbidden
assert any("artifacts:/app/artifacts:ro" in v for v in api["volumes"])   # read-only mount

services         : ['api', 'nginx', 'mlflow-ui']
always-on        : ['api'] (should be just 'api')
forbidden svcs   : none
api volumes      : ['./artifacts:/app/artifacts:ro', './data:/app/data:ro', './logs:/app/logs']
api healthcheck  : import urllib.request,sys; sys.exit(0 if urllib.request.urlo ...


## 3 — `.dockerignore` keeps the build context tiny

The image COPYs only code; the 900 MB of artifacts/data (and the local venv) are excluded — so the
daemon uploads megabytes, not gigabytes.

In [4]:
full = subprocess.run(["du", "-sh", "."], capture_output=True, text=True).stdout.split()[0]
ctx = subprocess.run(["du", "-sch", "src", "configs", "scripts", "requirements.txt"],
                     capture_output=True, text=True).stdout.strip().splitlines()[-1].split()[0]
print(f"full repo on disk          : {full}")
print(f"docker build context (COPY): {ctx}")
print("=> .dockerignore excludes artifacts/, data/, .venv/, .git/, ... (mounted at runtime, not baked)")

full repo on disk          : 4.0G
docker build context (COPY): 3.1M
=> .dockerignore excludes artifacts/, data/, .venv/, .git/, ... (mounted at runtime, not baked)


## 4 — Environment configuration (12-factor, no secrets)

`deployment_settings()` reads deployment env vars with safe defaults and **fails fast** on bad values.
There are NO secrets/API keys — the platform is deterministic with no external services.

In [5]:
from src.utils.config import deployment_settings
print("defaults :", deployment_settings({}))
try:
    deployment_settings({"LOG_LEVEL": "LOUD"})
except ValueError as e:
    print("validation:", e)
assert not any(k in str(deployment_settings({})).lower() for k in ("secret", "api_key", "token"))
print("no secrets in settings ✅")

defaults : {'port': 8000, 'workers': 1, 'log_level': 'INFO', 'log_file': None, 'deploy_mode': 'production', 'artifacts_dir': 'artifacts', 'data_dir': 'data', 'cors_origins': []}
validation: LOG_LEVEL must be one of ['CRITICAL', 'DEBUG', 'ERROR', 'INFO', 'WARNING'], got 'LOUD'
no secrets in settings ✅


## 5 — Health probes (LIVE via TestClient)

Three probes, three jobs: `/live` (process up), `/ready` (loaded → 200, else 503), `/health`
(diagnostics + resources). *(This boots the full warm stack — ~tens of seconds.)*

In [6]:
from fastapi.testclient import TestClient
from src.api.main import app
with TestClient(app) as c:
    print("/live  :", c.get("/live").status_code, c.get("/live").json())
    print("/ready :", c.get("/ready").status_code, c.get("/ready").json())
    h = c.get("/health").json()
    print("/health:", h["status"], "| system:",
          {k: h["system"][k] for k in ("process_rss_mb", "memory_percent", "disk_free_gb")})
    # readiness returns 503 while loading (simulate not-ready)
    c.app.state.registry = None
    print("/ready (loading):", c.get("/ready").status_code, c.get("/ready").json())

/Users/Yash/AI-Ventutre-Intelligence-Platform/.venv/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


2026-08-13 19:41:33 | avip.api                         | INFO     | API starting — OMP_NUM_THREADS=1, log_file=None, level=INFO


2026-08-13 19:41:34 | avip.core.registry               | INFO     | PredictionEngine ready — model=calibrated_model.joblib, threshold=0.329.


2026-08-13 19:41:35 | avip.core.registry               | INFO     | Item frame: 265171 campaigns, 13 item features, base funding rate 0.4038.


2026-08-13 19:41:35 | avip.core.registry               | INFO     | CounterfactualPipeline ready (warm model + item frame).


2026-08-13 19:41:35 | avip.core.registry               | INFO     | Loaded LGBMClassifier (865 features).


... (24 outputs left out)

/ready : 200 {'status': 'ready'}
2026-08-13 19:41:43 | avip.api                         | INFO     | [f7000156] GET /health -> 200 (3ms)


/health: ok | system: {'process_rss_mb': 1062.2, 'memory_percent': 84.5, 'disk_free_gb': 11.01}
2026-08-13 19:41:43 | avip.api                         | INFO     | [a4bd5316] GET /ready -> 503 (0ms)


2026-08-13 19:41:43 | avip.api                         | INFO     | [c4e742de] GET /ready -> 503 (0ms)


/ready (loading): 503 {'status': 'loading'}
2026-08-13 19:41:43 | avip.api                         | INFO     | API shutting down.


## 6 — Centralized logging

`configure_logging()` attaches one shared file sink to every layer's logger (existing + future).

In [7]:
import tempfile
from pathlib import Path
from src.utils.logger import get_logger, configure_logging
tmp = Path(tempfile.mkdtemp()) / "app.log"
res = configure_logging(log_file=str(tmp), level="INFO")
get_logger("avip.demo").info("a deployment log line")
import logging; logging.getLogger("avip.demo").handlers  # flush
print("configure_logging:", res)
print("log file line:", tmp.read_text().strip().split(" | ")[-1] if tmp.exists() else "(none)")

2026-08-13 19:41:43 | avip.demo                        | INFO     | a deployment log line


configure_logging: {'log_file': '/var/folders/jf/k4_k4dk97gg4hjz2674038dw0000gq/T/tmpqbtad_6j/app.log', 'level': 'INFO', 'loggers_reconfigured': 8}
log file line: a deployment log line


## 7 — CORS security (deny by default; never wildcard-with-credentials)

Empty `CORS_ORIGINS` = no cross-origin (secure default). An explicit allow-list enables credentials; a
bare `*` never does.

In [8]:
from src.utils.config import deployment_settings
def cors(origins):
    o = deployment_settings({"CORS_ORIGINS": origins} if origins is not None else {})["cors_origins"]
    creds = bool(o) and o != ["*"]
    return {"allow_origins": o, "allow_credentials": creds}
print("default (unset)     :", cors(None))
print("explicit allow-list :", cors("https://app.example.com"))
print("wildcard            :", cors("*"))

default (unset)     : {'allow_origins': [], 'allow_credentials': False}
explicit allow-list : {'allow_origins': ['https://app.example.com'], 'allow_credentials': True}
wildcard            : {'allow_origins': ['*'], 'allow_credentials': False}


## Summary — the pattern, and the honesty

**Deployment = package the process + its exact dependency closure into one reproducible container.** For
this platform that's a single lean image (13-layer monolith, FAISS in-process), artifacts **mounted**
not copied, environment pinned (`OMP_NUM_THREADS=1`, CPU-only torch), three health probes for
orchestrators, centralized file logging, a deny-by-default CORS allow-list, and an honest cloud blueprint
that admits the image is multi-GB. Everything above was validated **without Docker** — the one thing this
environment can't run, and the notebook says so plainly.

> **Interview soundbite.** *"Day 18 containerizes the platform as one lean image — FAISS is in-process,
> so it's a monolith, not microservices — with the 900 MB of artifacts mounted rather than baked in, and
> CPU-only torch to shrink a multi-gigabyte image. I added three health probes because liveness and
> readiness answer opposite questions during the model warm-up, centralized logging with one shared file
> sink, and a deny-by-default CORS allow-list that never does wildcard-with-credentials. Docker wasn't
> available in my environment, so I statically validated everything — the compose shape, the build
> context shrinking from 4 GB to 3 MB, the config and probes live via TestClient — and I'm explicit that
> the actual docker build is the operator's step, never claimed as run."*